# post_pipeline_verification

**Source:** `06_analysis/post_pipeline_verification.py`  
**Purpose:** Databricks notebook auto-generated from framework Python module.


## Section 1: Additional module code and configuration

This cell handles: *Additional module code and configuration*


In [ ]:
"""Post-pipeline verification queries for Databricks ingestion framework."""

from __future__ import annotations


## Section 2: Define `_q()` helper function

This cell handles: *Define `_q()` helper function*


In [ ]:
def _q(sql: str) -> str:
    return " ".join(sql.strip().split())


## Section 3: Define `run_post_pipeline_verification()` function with logic for processing

This cell handles: *Define `run_post_pipeline_verification()` function with logic for processing*


In [ ]:
def run_post_pipeline_verification(
    spark,
    catalog: str = "eng511_development_bronze",
    bronze_schema: str = "akr_tst",
    silver_schema: str = "silver",
    audit_schema: str = "audit",
    control_schema: str = "control",
):
    """Run verification queries and return a dict of DataFrames.

    Intended usage in Databricks notebook:
        results = run_post_pipeline_verification(spark)
        display(results["pipeline_runs_latest"])
    """

    queries = {
        "control_row_counts": _q(
            f"""
            SELECT 'source_registry' AS table_name, COUNT(*) AS row_count
            FROM {catalog}.{control_schema}.source_registry
            UNION ALL
            SELECT 'column_mapping', COUNT(*)
            FROM {catalog}.{control_schema}.column_mapping
            UNION ALL
            SELECT 'dq_rules', COUNT(*)
            FROM {catalog}.{control_schema}.dq_rules
            UNION ALL
            SELECT 'publish_rules', COUNT(*)
            FROM {catalog}.{control_schema}.publish_rules
            """
        ),
        "control_source_registry_detail": _q(
            f"DESCRIBE DETAIL {catalog}.{control_schema}.source_registry"
        ),
        "pipeline_runs_latest": _q(
            f"""
            SELECT run_id, source_system, source_entity, status, rows_in, rows_out, run_ts
            FROM {catalog}.{audit_schema}.pipeline_runs
            ORDER BY run_ts DESC
            LIMIT 50
            """
        ),
        "dq_results_latest": _q(
            f"""
            SELECT run_id, source_system, source_entity, dq_status, dq_failed_rule, row_count, run_ts
            FROM {catalog}.{audit_schema}.dq_rule_results
            ORDER BY run_ts DESC
            LIMIT 100
            """
        ),
        "connect_landing_distinct_files": _q(
            f"""
            SELECT COUNT(DISTINCT _source_file) AS distinct_files_picked
            FROM {catalog}.{bronze_schema}.connect_countryriskdet_landing
            WHERE _source_file LIKE '%/deltaload/%'
            """
        ),
        "connect_landing_files_breakdown": _q(
            f"""
            SELECT _source_file, COUNT(*) AS rows_per_file
            FROM {catalog}.{bronze_schema}.connect_countryriskdet_landing
            WHERE _source_file LIKE '%/deltaload/%'
            GROUP BY _source_file
            ORDER BY rows_per_file DESC
            """
        ),
        "silver_connect_row_count": _q(
            f"SELECT COUNT(*) AS silver_row_count FROM {catalog}.{silver_schema}.connect_countryriskdet"
        ),
        "lifecycle_log_latest": _q(
            f"""
            SELECT entity_id, event_type, event_ts
            FROM {catalog}.{audit_schema}.entity_lifecycle_log
            ORDER BY event_ts DESC
            LIMIT 50
            """
        ),
    }

    results = {}
    for name, query in queries.items():
        results[name] = spark.sql(query)

    return results
